# Problem 1 — Line Fitting

**Dataset:** `../data/lines.csv` — 102 points (x1,x2,x3, y1,y2,y3) representing 3 mixed line groups.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

data = np.loadtxt('../data/lines.csv', delimiter=',')
print('Shape:', data.shape)
print(data[:5])

In [ ]:
# Unpack the three groups of (x, y) point pairs
x1, x2, x3 = data[:, 0], data[:, 1], data[:, 2]
y1, y2, y3 = data[:, 3], data[:, 4], data[:, 5]

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(x1, y1, label='Line 1')
ax.scatter(x2, y2, label='Line 2')
ax.scatter(x3, y3, label='Line 3')
ax.legend()
ax.set_title('Raw data points')
plt.show()

## Part (a) — Total Least Squares on Line 1

Fit a single line to the **first group only** using Total Least Squares (orthogonal regression via SVD).

Line represented in normal form: $ax + by = d$, where $(a,b)$ is the unit normal.

In [ ]:
# --- Total Least Squares via SVD ---
X1 = np.column_stack((x1, y1))   # shape (N, 2)

U = X1 - np.mean(X1, axis=0)              # centre the data
_, _, Vt = np.linalg.svd(U.T @ U)         # SVD of scatter matrix
a, b = Vt[-1, 0], Vt[-1, 1]               # normal = last row of Vt
d = a * np.mean(X1[:, 0]) + b * np.mean(X1[:, 1])

print(f'Line (normal form): {a:.4f}x + {b:.4f}y = {d:.4f}')

# Plot
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(x1, y1, label='Line 1 points')

x_range = np.array([x1.min() - 1, x1.max() + 1])
y_fit = (d - a * x_range) / b
ax.plot(x_range, y_fit, 'r-', linewidth=2, label='TLS fit')

ax.legend()
ax.set_title('Part (a): Total Least Squares')
ax.set_aspect('equal')
plt.tight_layout()
plt.savefig('../output/p1a_tls.png', dpi=150)
plt.show()

## Part (b) — RANSAC on All Three Lines

Fit **3 separate lines** from all 102 mixed points using RANSAC.
Each iteration: sample 2 points → compute candidate line → count inliers → refit with inliers.

In [ ]:
import math
from scipy.optimize import minimize

# All 306 points (stack all three groups)
all_x = np.concatenate([x1, x2, x3])
all_y = np.concatenate([y1, y2, y3])
X_all = np.column_stack((all_x, all_y))
N = len(X_all)
print('Total points:', N)

In [ ]:
def line_from_two_points(p1, p2):
    """Return unit-normal line (a, b, d) through two points."""
    dx, dy = p2[0] - p1[0], p2[1] - p1[1]
    mag = math.sqrt(dx**2 + dy**2)
    a, b = dy / mag, -dx / mag
    d = a * p1[0] + b * p1[1]
    return np.array([a, b, d])

def inliers(X, model, t):
    """Boolean mask of points within distance t from line ax+by=d."""
    a, b, d = model
    return np.abs(a * X[:, 0] + b * X[:, 1] - d) < t

def fit_tls(X):
    """Fit a line to point set X using TLS/SVD."""
    U = X - np.mean(X, axis=0)
    _, _, Vt = np.linalg.svd(U.T @ U)
    a, b = Vt[-1, 0], Vt[-1, 1]
    d = a * np.mean(X[:, 0]) + b * np.mean(X[:, 1])
    return np.array([a, b, d])

In [ ]:
def ransac_line(X, t=0.5, min_inliers=0.25, max_iter=200):
    """RANSAC line fitting. Returns (model, inlier_mask)."""
    N = len(X)
    best_model, best_mask = None, None
    best_count = 0

    for _ in range(max_iter):
        idx = np.random.choice(N, 2, replace=False)
        candidate = line_from_two_points(X[idx[0]], X[idx[1]])
        mask = inliers(X, candidate, t)

        if mask.sum() > best_count:
            # Refit using all inliers
            if mask.sum() >= 2:
                refined = fit_tls(X[mask])
                mask2 = inliers(X, refined, t)
                if mask2.sum() > best_count:
                    best_model = refined
                    best_mask  = mask2
                    best_count = mask2.sum()

    return best_model, best_mask

In [ ]:
np.random.seed(42)

remaining = np.ones(N, dtype=bool)   # mask of points not yet claimed
lines     = []
masks     = []
colors    = ['red', 'blue', 'green']

for i in range(3):
    model, mask_local = ransac_line(X_all[remaining], t=0.5, max_iter=300)
    # Map local mask back to global indices
    global_idx = np.where(remaining)[0]
    global_mask = np.zeros(N, dtype=bool)
    global_mask[global_idx[mask_local]] = True

    lines.append(model)
    masks.append(global_mask)
    remaining[global_mask] = False
    print(f'Line {i+1}: {model} | inliers: {global_mask.sum()}')

print('Unclaimed points:', remaining.sum())

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))

# Plot inliers per line
for i, (model, mask, col) in enumerate(zip(lines, masks, colors)):
    ax.scatter(X_all[mask, 0], X_all[mask, 1], color=col,
               label=f'Line {i+1} inliers', zorder=3)
    a, b, d = model
    xs = np.array([X_all[mask, 0].min() - 1, X_all[mask, 0].max() + 1])
    ys = (d - a * xs) / b
    ax.plot(xs, ys, color=col, linewidth=2)

# Outliers
outlier_mask = ~(masks[0] | masks[1] | masks[2])
ax.scatter(X_all[outlier_mask, 0], X_all[outlier_mask, 1],
           color='gray', alpha=0.5, label='Outliers', zorder=2)

ax.legend()
ax.set_title('Part (b): RANSAC — 3 Lines')
ax.set_aspect('equal')
plt.tight_layout()
plt.savefig('../output/p1b_ransac.png', dpi=150)
plt.show()